This solution uses a two-stage ensemble approach by properly handling the distribution shift in the competition data.

Critical insight: optimizing on full CV (dominated by pre-cutoff data) can hurt LB performance. Instead, we optimize only on post-cutoff CV, which correlates with the leaderboard.

Two-Stage Pipeline
Stage 1: Hill Climbing Model Selection
Method: Greedy forward selection with rank-based blending.
Optimization target: Post-cutoff CV AUC (not full CV).
Features:
Rank transformation normalizes models with different scales.
Negative weights enabled (-0.3 to +0.5) for better diversity.
GPU-accelerated AUC computation for fast iterations.
Selects ~60 best models through iterative improvement.

Stage 2: Ridge Ensemble Stacking
Input: Top-36 models from Hill Climbing selection.
Method: Ridge regression (alpha=10.0) on rank-transformed predictions.
Training: Fitted on post-cutoff data only.

Why this works well: rank transformation handles models with different output scales. Hill Climbing identifies synergistic model combinations. Ridge stacking provides stable, regularized blending weights. Two-stage approach combines exploration (HC) with exploitation (Ridge). 

Diversity. The solution leverages ~100 diverse base models including:
XGBoost, LightGBM, CatBoost with various hyperparameters;
Neural networks (MLP, FastAI, PyTorch);
Ensemble methods (Ridge, weighted blending, AutoML);
Different feature engineering approaches;
Multiple seeds and cross-validation strategies.

Results
Post-Cutoff CV: 0.708600 (optimized metric)
Public LB: 0.70739 (correlates with post-cutoff CV)
Private LB: 0.70504 (1st place 🏆)

In [ ]:
import os, time, warnings
import numpy as np
import pandas as pd
from datetime import datetime
warnings.filterwarnings("ignore")

import torch
from sklearn.linear_model import Ridge

print("="*80)
print("1ST PLACE SOLUTION - HILL CLIMBING + RIDGE ENSEMBLE")
print("="*80 + "\n")

# =============================================================================
# CONFIGURATION
# =============================================================================

# Distribution shift cutoff
CUTOFF_ID = 678260

# Hill Climbing parameters
HC_BLENDING_METHOD = "rank"  # "rank" or "probability"
HC_ALLOW_NEGATIVE = True
HC_WEIGHT_MIN = -0.30 if HC_ALLOW_NEGATIVE else 0.0
HC_WEIGHT_MAX = 0.50
HC_WEIGHT_STEP = 0.01
HC_TOLERANCE = 1e-7

# Starting model for Hill Climbing
# Options: None (auto-select best), int (model index), or str (model name)
HC_START_MODEL = '19122025_XGB_leakage_solved_5folds_3seeds'  # Set to model name or index to override

# Ridge Ensemble parameters
RIDGE_TOP_N = 36
RIDGE_ALPHA = 10.0

# Paths
TARGET_PATH = "/kaggle/input/playground-series-s5e12/train.csv"
TEST_PATH = "/kaggle/input/playground-series-s5e12/test.csv"
TARGET_COLUMN = "diagnosed_diabetes"
OUT_DIR = "/kaggle/working"

# =============================================================================
# MODEL LIST - ADD YOUR MODELS HERE
# =============================================================================

MODELS = [
    {
        'name': '10122025_Ridge_Ensemble_2_3ways_combos',
        'oof_path': "/kaggle/input/oof-diabetic/20251210_133339_XGB_LGB_CB_RidgeStack_GPU_3WayTE500_AUC0p729461_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251210_133339_XGB_LGB_CB_RidgeStack_GPU_3WayTE500_AUC0p729461_test.npy",
    },
    {
        'name': '13122025_HC_with_Ridge',
        'oof_path': "/kaggle/input/oof-diabetic/ensemble_oof_greedy_rank_13of14_neg_auc0.73204_20251213_083230.npy",
        'test_path': "/kaggle/input/oof-diabetic/ensemble_test_greedy_rank_13of14_neg_auc0.73204_20251213_083230.npy",
    },
    {
        'name': '13122025_HC_with_Ridge_v1',
        'oof_path': "/kaggle/input/oof-diabetic/ensemble_oof_greedy_rank_7of9_neg_auc0.73092_20251213_085621.npy",
        'test_path': "/kaggle/input/oof-diabetic/ensemble_test_greedy_rank_7of9_neg_auc0.73092_20251213_085621.npy",
    },
    {
        'name': '10122025_Ridge_Ensemble_2ways_combos',
        'oof_path': "/kaggle/input/oof-diabetic/20251209_204356_XGB_LGB_CB_RidgeStack_AUC0p729581_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251209_204356_XGB_LGB_CB_RidgeStack_AUC0p729581_test.npy",
    },
    {
        'name': '01122025_XGB_Masaya_approach',
        'oof_path': "/kaggle/input/oof-diabetic/01122025_oof_XGB_masaya_0.73065.npy",
        'test_path': "/kaggle/input/oof-diabetic/01122025_test_XGB_masaya_0.73065.npy",
    },
    {
        'name': '01122025_XGB',
        'oof_path': "/kaggle/input/oof-diabetic/20251201_140206_XGBoost_CV0p7304_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251201_140206_XGBoost_CV0p7304_test_preds.npy",
    },
    {
        'name': '01122025_LightGBM',
        'oof_path': "/kaggle/input/oof-diabetic/20251201_142349_LightGBM_CV0p7272_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251201_142349_LightGBM_CV0p7272_test_preds.npy",
    },
    {
        'name': '01122025_XGB_Orig_as_columns',
        'oof_path': "/kaggle/input/oof-diabetic/20251201_150639_XGB1C_AUC_0_722554_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251201_150639_XGB1C_AUC_0_722554_test_preds.npy",
    },
    {
        'name': '01122025_Ensemble',
        'oof_path': "/kaggle/input/oof-diabetic/20251201_150832_ENS_LGB_XGB_AUC_0_727766_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251201_150832_ENS_LGB_XGB_AUC_0_727766_test_preds.npy",
    },
    {
        'name': '01122025_XGB',
        'oof_path': "/kaggle/input/oof-diabetic/oof_preds_xgb_20251201_162314_auc0p730998.npy",
        'test_path': "/kaggle/input/oof-diabetic/test_preds_xgb_20251201_162314_auc0p730998.npy",
    },
    {
        'name': '01122025_HisGradientBoosting',
        'oof_path': "/kaggle/input/oof-diabetic/20251201_183723_HistGB_CV0p7244_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251201_183723_HistGB_CV0p7244_test_preds.npy",
    },
    {
        'name': '01122025_LightGBM_optimized',
        'oof_path': "/kaggle/input/oof-diabetic/20251201_185354_LightGBM_CV0p7276_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251201_185354_LightGBM_CV0p7276_test_preds.npy",
    }, 
    {
        'name': '01122025_LightGBM_optimized_v1',
        'oof_path': "/kaggle/input/oof-diabetic/20251201_194126_LightGBM_Optuna_CV0p7286_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251201_194126_LightGBM_Optuna_CV0p7286_test_preds.npy",
    },
    {
        'name': '01122025_3Models_Ridge_Ensemble',
        'oof_path': "/kaggle/input/oof-diabetic/20251201_202841_XGB_LGB_CB_RidgeStack_AUC0p730406_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251201_202841_XGB_LGB_CB_RidgeStack_AUC0p730406_test.npy",
    },
    {
        'name': '01122025_LightGBM_optimized_v2',
        'oof_path': "/kaggle/input/oof-diabetic/20251201_201939_LightGBM_Optuna_CV0p7290_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251201_201939_LightGBM_Optuna_CV0p7290_test_preds.npy",
    },
    {
        'name': '02122025_HistGradient_improved',
        'oof_path': "/kaggle/input/oof-diabetic/20251202_060231_HistGradientBoosting_01_baseline_AUC0_724711_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251202_060231_HistGradientBoosting_01_baseline_AUC0_724711_test.npy",
    }, 
    {
        'name': '02122025_LightGBM_optimized_poly_features',
        'oof_path': "/kaggle/input/oof-diabetic/20251202_072811_LightGBM_Optuna_CV0p7293_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251202_072811_LightGBM_Optuna_CV0p7293_test_preds.npy",
    }, 
    {
        'name': '02122025_HistGradient_improved_quantile_difference_features',
        'oof_path': "/kaggle/input/oof-diabetic/20251202_060231_HistGradientBoosting_15_quantile_diff_AUC0_724887_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251202_060231_HistGradientBoosting_15_quantile_diff_AUC0_724887_test.npy",
    }, 
    {
        'name': '02122025_MLP_v1',
        'oof_path': "/kaggle/input/oof-diabetic/20251202_142022_Keras_MLP_CV0p6970_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251202_142022_Keras_MLP_CV0p6970_test_preds.npy",
    }, 
    {
        'name': '02122025_MLP_v2',
        'oof_path': "/kaggle/input/oof-diabetic/20251202_144355_Keras_DeepMLP_CV0p6970_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251202_144355_Keras_DeepMLP_CV0p6970_test_preds.npy",
    }, 
    {
        'name': '02122025_XGB_Optuna',
        'oof_path': "/kaggle/input/oof-diabetic/20251202_170729_XGBoost_Optuna_GPU_CV0p7283_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251202_170729_XGBoost_Optuna_GPU_CV0p7283_test_preds.npy",
    },
    {
        'name': '02122025_LightGBM',
        'oof_path': "/kaggle/input/oof-diabetic/20251202_175949_LightGBM_AdvancedFE_CV0p7283_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251202_175949_LightGBM_AdvancedFE_CV0p7283_test_preds.npy",
    },
    {
        'name': '02122025_MLP_PyTorch',
        'oof_path': "/kaggle/input/oof-diabetic/20251202_160809_PyTorch_MLP_CV0p6968_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251202_160809_PyTorch_MLP_CV0p6968_test.npy",
    }, 
    {
        'name': '03122025_LightGBM_Log_Features_Optuna',
        'oof_path': "/kaggle/input/oof-diabetic/20251203_082827_LightGBM_Optuna_LogFE_CV0p7277_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251203_082827_LightGBM_Optuna_LogFE_CV0p7277_test_preds.npy",
    },
    # {
    #     'name': '03122025_CatBoost_LightAutoML',
    #     'oof_path': "/kaggle/input/oof-diabetic/oof_LightAutoML_CatBoost_20251203_110046_AUC072833.npy",
    #     'test_path': "/kaggle/input/oof-diabetic/test_LightAutoML_CatBoost_20251203_110046_AUC072833.npy",
    # },
    {
        'name': '03122025_LightAutoML_CB_XGB_LGBM_ensemble',
        'oof_path': "/kaggle/input/oof-diabetic/oof_LightAutoML_CB_XGB_LGB_20251203_114203_AUC072919.npy",
        'test_path': "/kaggle/input/oof-diabetic/test_LightAutoML_CB_XGB_LGB_20251203_114203_AUC072919.npy",
    },
    {
        'name': '03122025_XGB_OrigsAsColumns_v1',
        'oof_path': "/kaggle/input/oof-diabetic/20251203_151544_XGB1C_CV0.730444_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251203_151544_XGB1C_CV0.730444_test.npy",
    },
    {
        'name': '03122025_XGB_OrigsAsColumns_v2',
        'oof_path': "/kaggle/input/oof-diabetic/20251203_154721_XGB1C_CV0.730808_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251203_154721_XGB1C_CV0.730808_test.npy",
    },
    {
        'name': '03122025_LightGBM_PolyFeatures_Optuna',
        'oof_path': "/kaggle/input/oof-diabetic/20251203_123814_LightGBM_Optuna_CV0p7292_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251203_123814_LightGBM_Optuna_CV0p7292_test_preds.npy",
    },
    {
        'name': '03122025_LightGBM_PolyInteractionFeatures_Optuna',
        'oof_path': "/kaggle/input/oof-diabetic/20251203_123402_LightGBM_Optuna_CV0p7284_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251203_123402_LightGBM_Optuna_CV0p7284_test_preds.npy",
    },
    {
        'name': '04122025_3models_Ridge_no_original_data',
        'oof_path': "/kaggle/input/oof-diabetic/20251204_064438_XGB_LGB_CB_RidgeStack_AUC0p726898_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251204_064438_XGB_LGB_CB_RidgeStack_AUC0p726898_test.npy",
    },
    {
        'name': '05122025_3models_ensemble',
        'oof_path': "/kaggle/input/oof-diabetic/20251205_ensemble_3models_AUC0.726402_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251205_ensemble_3models_AUC0.726402_test.npy",
    },
    {
        'name': '05122025_XGB_ratio_features_Optuna_optimized',
        'oof_path': "/kaggle/input/oof-diabetic/20251205_062956_XGBoost_Optuna_CV0p7277_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251205_062956_XGBoost_Optuna_CV0p7277_test_preds.npy",
    },
    {
        'name': '05122025_3models_Ensemble_ratio_features',
        'oof_path': "/kaggle/input/oof-diabetic/20251205_ensemble_3models_AUC0.726498_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251205_ensemble_3models_AUC0.726498_test.npy",
    },
    {
        'name': '05122025_3models_Ensemble_rank_based_additional_hyperparams',
        'oof_path': "/kaggle/input/oof-diabetic/20251205_lgb_rank_ensemble_5models_AUC0.725260_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251205_lgb_rank_ensemble_5models_AUC0.725260_test.npy",
    },
    {
        'name': '05122025_3models_Ensemble_simple_average_additional_hyperparams',
        'oof_path': "/kaggle/input/oof-diabetic/20251205_lgb_rank_ensemble_5models_AUC0.725260_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251205_lgb_rank_ensemble_5models_AUC0.725260_test.npy",
    },
    {
        'name': '05122025_LightGBM_baseline',
        'oof_path': "/kaggle/input/oof-diabetic/20251205_lgb_v1_baseline_AUC0.725692_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251205_lgb_v1_baseline_AUC0.725692_test.npy",
    },
    {
        'name': '05122025_LightGBM_deep',
        'oof_path': "/kaggle/input/oof-diabetic/20251205_lgb_v2_deep_AUC0.726194_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251205_lgb_v2_deep_AUC0.726194_test.npy",
    },
    {
        'name': '05122025_LightGBM_dart',
        'oof_path': "/kaggle/input/oof-diabetic/20251205_lgb_v3_dart_AUC0.723196_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251205_lgb_v3_dart_AUC0.723196_test.npy",
    },
    {
        'name': '05122025_LightGBM_goss',
        'oof_path': "/kaggle/input/oof-diabetic/20251205_lgb_v4_goss_AUC0.720153_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251205_lgb_v4_goss_AUC0.720153_test.npy",
    },
    {
        'name': '05122025_LightGBM_low_LR',
        'oof_path': "/kaggle/input/oof-diabetic/20251205_lgb_v5_lowlr_AUC0.724871_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251205_lgb_v5_lowlr_AUC0.724871_test.npy",
    },
    {
        'name': '05122025_CatBoost',
        'oof_path': "/kaggle/input/oof-diabetic/20251205_143805_CatBoost_AUC0p724244_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251205_143805_CatBoost_AUC0p724244_test_preds.npy",
    },
    {
        'name': '05122025_3models_ensemble',
        'oof_path': "/kaggle/input/oof-diabetic/20251205_154004_Ensemble_Weighted_AUC0p727369_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251205_154004_Ensemble_Weighted_AUC0p727369_test.npy",
    },
    {
        'name': '05122025_XGB_depth2',
        'oof_path': "/kaggle/input/oof-diabetic/oof_XGBoost_20251205_151958_AUC0_726204.npy",
        'test_path': "/kaggle/input/oof-diabetic/test_XGBoost_20251205_151958_AUC0_726204.npy",
    },
    {
        'name': '06122025-YDF_v1',
        'oof_path': "/kaggle/input/oof-diabetic/oof_YDF_GradientBoostedTrees_20251205_163434_AUC0_724446.npy",
        'test_path': "/kaggle/input/oof-diabetic/test_YDF_GradientBoostedTrees_20251205_163434_AUC0_724446.npy",
    },
    {
        'name': '06122025_XGB_v1',
        'oof_path': "/kaggle/input/oof-diabetic/20251205_183433_XGBoost_AUC_0-725915_oof_predictions.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251205_183433_XGBoost_AUC_0-725915_test_predictions.npy",
    },
    {
        'name': '06122025_XGB_v2',
        'oof_path': "/kaggle/input/oof-diabetic/20251205_184511_XGBoost_AUC_0-727396_oof_predictions.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251205_184511_XGBoost_AUC_0-727396_test_predictions.npy",
    },
    {
        'name': '06122025_YDF',
        'oof_path': "/kaggle/input/oof-diabetic/20251206_ydf_gbt_v1_AUC0.724666_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251206_ydf_gbt_v1_AUC0.724666_test.npy",
    },
    {
        'name': '06122025_Experimental_3models_ensemble',
        'oof_path': "/kaggle/input/oof-diabetic/20251206_140942_Ensemble_AUC072697_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251206_140942_Ensemble_AUC072697_test.npy",
    },
    {
        'name': '06122025_Ridge_3models_ensemble_selective_origdata',
        'oof_path': "/kaggle/input/oof-diabetic/20251206_143738_XGB_LGB_CB_RidgeStack_V2_AUC0p726486_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251206_143738_XGB_LGB_CB_RidgeStack_V2_AUC0p726486_test.npy",
    },
    {
        'name': '06122025_CatBoost_proxy_missing_feature',
        'oof_path': "/kaggle/input/oof-diabetic/20251206_152328_CatBoost_AUC072775_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251206_152328_CatBoost_AUC072775_test.npy",
    },
    {
        'name': '07122025_CatBoost',
        'oof_path': "/kaggle/input/oof-diabetic/07122025_cb_oof_preds_tarter.npy",
        'test_path': "/kaggle/input/oof-diabetic/07122025_cb_test_preds_tarter.npy",
    },
    {
        'name': '07122025_CatBoost_proxy_advanced',
        'oof_path': "/kaggle/input/oof-diabetic/20251207_072907_CatBoost_AUC072767_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251207_072907_CatBoost_AUC072767_test.npy",
    },
    {
        'name': '07122025_XGB_proxy_advanced_v1',
        'oof_path': "/kaggle/input/oof-diabetic/20251207_123702_XGBoost_AUC072648_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251207_123702_XGBoost_AUC072648_test.npy",
    },
    {
        'name': '07122025_XGB_interaction_features',
        'oof_path': "/kaggle/input/oof-diabetic/20251207_144949_XGBoost_Interactions_AUC072349_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251207_144949_XGBoost_Interactions_AUC072349_test.npy",
    },
    {
        'name': '07122025_CatBoost_advanced_proxy',
        'oof_path': "/kaggle/input/oof-diabetic/20251207_152718_CatBoost_AUC072779_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251207_152718_CatBoost_AUC072779_test.npy",
    },
    {
        'name': '07122025_XGB',
        'oof_path': "/kaggle/input/oof-diabetic/20251207_170634_xgb_cv072720_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251207_170634_xgb_cv072720_test.npy",
    },
    {
        'name': '07122025_4models_ensemble',
        'oof_path': "/kaggle/input/oof-diabetic/20251207_194837_oof_ensemble_Claude_0.727325.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251207_194837_test_ensemble_Claude_0.727325.npy",
    },
    {
        'name': '08122025_XGB',
        'oof_path': "/kaggle/input/oof-diabetic/08122025_xgb_oof_tartler_0.69641.npy",
        'test_path': "/kaggle/input/oof-diabetic/08122025_xgb_test_tartler_0.69641.npy",
    },
    {
        'name': '09122025_LGBM_Target_Encoded',
        'oof_path': "/kaggle/input/oof-diabetic/20251209_074546_lgbm_te_cv0p72815_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251209_074546_lgbm_te_cv0p72815_test.npy",
    },
    {
        'name': '07122025_LGBM_from_ensemble',
        'oof_path': "/kaggle/input/oof-diabetic/20251207_194837_oof_lightgbm.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251207_194837_test_lightgbm.npy",
    },
    {
        'name': '07122025_CatBoost_from_ensemble',
        'oof_path': "/kaggle/input/oof-diabetic/20251207_194837_oof_catboost.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251207_194837_test_catboost.npy",
    },
    {
        'name': '07122025_XGB_from_ensemble',
        'oof_path': "/kaggle/input/oof-diabetic/20251207_194837_oof_xgboost.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251207_194837_test_xgboost.npy",
    },
    {
        'name': '09122025_LGBM_TE_Numerical',
        'oof_path': "/kaggle/input/oof-diabetic/20251209_092445_lgbm_te_full_cv0p72776_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251209_092445_lgbm_te_full_cv0p72776_test.npy",
    },
    {
        'name': '09122025_XGB_BB_Optimized',
        'oof_path': "/kaggle/input/oof-diabetic/20251209_101001_XGBoost_Optuna_GPU_CV0p729979_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251209_101001_XGBoost_Optuna_GPU_CV0p729979_test.npy",
    },
    {
        'name': '09122025_SeedOptimized',
        'oof_path': "/kaggle/input/oof-diabetic/20251209_114517_blend_oof_auc0.72920.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251209_114517_blend_test_auc0.72920.npy",
    },
    {
        'name': '09122025_Ridge_3Models',
        'oof_path': "/kaggle/input/oof-diabetic/20251209_144158_XGB_LGB_CB_RidgeStack_AUC0p730777_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251209_144158_XGB_LGB_CB_RidgeStack_AUC0p730777_test.npy",
    }, 
    {
        'name': '09122025_Optuna_4_models_ensemble',
        'oof_path': "/kaggle/input/oof-diabetic/20251209_133608_XGB_LGB_LR_Stack_AUC_0_72946_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251209_133608_XGB_LGB_LR_Stack_AUC_0_72946_test.npy",
    }, 
    {
        'name': '04122025_LightGBM_Optuna_CV0p7284',
        'oof_path': "/kaggle/input/oof-diabetic/20251204_063452_LightGBM_Optuna_CV0p7284_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251204_063452_LightGBM_Optuna_CV0p7284_test_preds.npy",
    },
    {
        'name': '04122025_LightGBM_5Seed_Ratios_CV0p7290',
        'oof_path': "/kaggle/input/oof-diabetic/20251204_090655_LightGBM_5Seed_Ratios_CV0p7290_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251204_090655_LightGBM_5Seed_Ratios_CV0p7290_test_preds.npy",
    },
    {
        'name': '04122025_LightGBM_Optuna_CV0p7286',
        'oof_path': "/kaggle/input/oof-diabetic/20251204_130346_LightGBM_Optuna_CV0p7286_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251204_130346_LightGBM_Optuna_CV0p7286_test_preds.npy",
    },
    {
        'name': '10122025_Ensemble',
        'oof_path': "/kaggle/input/oof-diabetic/20251210_Ensemble_AUC0.72787_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251210_Ensemble_AUC0.72787_test.npy",
    },
    {
        'name': '11122025_CatBoost',
        'oof_path': "/kaggle/input/oof-diabetic/20251211_075716_CatBoost_CV0.73082_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251211_075716_CatBoost_CV0.73082_test.npy",
    },
    {
        'name': '11122025_CatBoost_Optimized',
        'oof_path': "/kaggle/input/oof-diabetic/20251211_145528_CatBoost_Optuna_CV0p7309_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251211_145528_CatBoost_Optuna_CV0p7309_test.npy",
    },
    {
        'name': '11122025_LightGBM',
        'oof_path': "/kaggle/input/oof-diabetic/20251211_091203_LightGBM_CV0.72950_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251211_091203_LightGBM_CV0.72950_test.npy",
    },
    {
        'name': '10122025_Ridge_3_models_600_TE_features',
        'oof_path': "/kaggle/input/oof-diabetic/20251210_172753_XGB_LGB_CB_RidgeStack_GPU_3WayTE600_AUC0p729442_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251210_172753_XGB_LGB_CB_RidgeStack_GPU_3WayTE600_AUC0p729442_test.npy",
    },
    {
        'name': '04122025_XGB_OriginalWeighted_CV0p7272',
        'oof_path': "/kaggle/input/oof-diabetic/20251204_160716_XGB_OriginalWeighted_CV0p7272_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251204_160716_XGB_OriginalWeighted_CV0p7272_test_preds.npy",
    },
    {
        'name': '06122025_LightGBM_AUC072956',
        'oof_path': "/kaggle/input/oof-diabetic/20251206_163253_LightGBM_AUC072956_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251206_163253_LightGBM_AUC072956_test.npy",
    },
    {
        'name': '06122025_LightGBM_AUC072906',
        'oof_path': "/kaggle/input/oof-diabetic/20251206_170406_LightGBM_AUC072906_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251206_170406_LightGBM_AUC072906_test.npy",
    },
    {
        'name': '06122025_LightGBM_AUC072923',
        'oof_path': "/kaggle/input/oof-diabetic/20251206_194324_LightGBM_AUC072923_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251206_194324_LightGBM_AUC072923_test.npy",
    },
    {
        'name': '06122025_LightGBM_AUC072959',
        'oof_path': "/kaggle/input/oof-diabetic/20251206_202311_LightGBM_AUC072959_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251206_202311_LightGBM_AUC072959_test.npy",
    },
    {
        'name': '06122025_CatBoost_AUC07278',
        'oof_path': "/kaggle/input/oof-diabetic/20251206_CatBoost_AUC07278_oof_preds.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251206_CatBoost_AUC07278_test_preds.npy",
    },
    {
        'name': '08122025_LightGBM_AUC0p727114',
        'oof_path': "/kaggle/input/oof-diabetic/20251208_084136_LightGBM_5Fold_AUC0p727114_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251208_084136_LightGBM_5Fold_AUC0p727114_test.npy",
    },
    {
        'name': '08122025_LightGBM_FE_5Fold_AUC0p726493',
        'oof_path': "/kaggle/input/oof-diabetic/20251208_090540_LightGBM_FE_5Fold_AUC0p726493_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251208_090540_LightGBM_FE_5Fold_AUC0p726493_test.npy",
    },
    {
        'name': '08122025_LightGBM_Optuna_AUC0p728862',
        'oof_path': "/kaggle/input/oof-diabetic/20251208_101349_LightGBM_Optuna_AUC0p728862_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251208_101349_LightGBM_Optuna_AUC0p728862_test.npy",
    },
    {
        'name': '08122025_LightAutoML_CB_EDA_AUC0p729347',
        'oof_path': "/kaggle/input/oof-diabetic/20251208_132826_LightAutoML_CB_EDA_AUC0p729347_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251208_132826_LightAutoML_CB_EDA_AUC0p729347_test.npy",
    },
    {
        'name': '08122025_LightAutoML_CatBoost_AUC0p729009',
        'oof_path': "/kaggle/input/oof-diabetic/20251208_155304_LightAutoML_CatBoost_AUC0p729009_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251208_155304_LightAutoML_CatBoost_AUC0p729009_test.npy",
    },
    {
        'name': '12122025_LightGBM_Freq_FE',
        'oof_path': "/kaggle/input/oof-diabetic/20251211_114805_LightGBM_EthnicityFreq_CV0p7294_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251211_114805_LightGBM_EthnicityFreq_CV0p7294_test.npy",
    },
    {
        'name': '11122025_CatBoost_2ways_combos',
        'oof_path': "/kaggle/input/oof-diabetic/20251211_165705_CatBoost_CV0.73081_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251211_165705_CatBoost_CV0.73081_test.npy",
    },
    {
        'name': '12122025_CatBoost_original_data',
        'oof_path': "/kaggle/input/oof-diabetic/12122025_CatBoost_Styrov_orig_data_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/12122025_CatBoost_Styrov_orig_data_test.npy",
    },
    {
        'name': '11122025_LGBM_CV0p7306',
        'oof_path': "/kaggle/input/oof-diabetic/20251212_043837_LightGBM_CV0p7306_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251212_043837_LightGBM_CV0p7306_test.npy",
    },
    {
        'name': '12122025_XGB_CV0p7306',
        'oof_path': "/kaggle/input/oof-diabetic/20251212_070931_XGBoost_Optuna_GPU_CV0p7306_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251212_070931_XGBoost_Optuna_GPU_CV0p7306_test.npy",
    },
    {
        'name': '12122025_Ridge_3modles_Advanced_Features_Selection',
        'oof_path': "/kaggle/input/oof-diabetic/20251212_104639_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251212_104639_test.npy",
    },
    {
        'name': '12122025_XGB_experimental',
        'oof_path': "/kaggle/input/oof-diabetic/20251212_173423_v5_xgb_oof_trainonly.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251212_173423_v5_xgb_test.npy",
    },
    {
        'name': '12122025_MLP_GB_ensemble',
        'oof_path': "/kaggle/input/oof-diabetic/20251212_194249_STACK_MLP_GBC_AUC0p7242_oof_predictions.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251212_194249_STACK_MLP_GBC_AUC0p7242_test_predictions.npy",
    },
    {
        'name': '13122025_LGBM_experimental',
        'oof_path': "/kaggle/input/oof-diabetic/oof_lgbm_qt_clean_0p72853_20251213_091536.npy",
        'test_path': "/kaggle/input/oof-diabetic/test_lgbm_qt_clean_0p72853_20251213_091536.npy",
    },
    {
        'name': '13122025_3stages_FE_ensemble',
        'oof_path': "/kaggle/input/oof-diabetic/oof_ensemble_weighted_0p72795_20251213_094042.npy",
        'test_path': "/kaggle/input/oof-diabetic/test_ensemble_weighted_0p72795_20251213_094042.npy",
    },
    {
        'name': '13122025_Catboost_AUC0p726084',
        'oof_path': "/kaggle/input/oof-diabetic/20251213_131918_CatBoost_GPU_AUC0p726084_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251213_131918_CatBoost_GPU_AUC0p726084_test.npy",
    },
    {
        'name': '14122025_Catboost_Improved',
        'oof_path': "/kaggle/input/oof-diabetic/20251213_200927_CatBoost_Improved_AUC0p726705_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251213_200927_CatBoost_Improved_AUC0p726705_test.npy",
    },
    {
        'name': '13122025_Ridge_Modified_by_Claude',
        'oof_path': "/kaggle/input/oof-diabetic/20251213_122933_ImprovedEnsemble_ErrorAnalysis_AUC0p727129_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251213_122933_ImprovedEnsemble_ErrorAnalysis_AUC0p727129_test.npy",
    },
    {
        'name': '13122025_Ridge_Modified_by_Claude_NoOriginalData',
        'oof_path': "/kaggle/input/oof-diabetic/20251213_141652_RevisedEnsemble_NoExtTE_PseudoLabel_AUC0p727122_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251213_141652_RevisedEnsemble_NoExtTE_PseudoLabel_AUC0p727122_test.npy",
    },
    {
        'name': '20251205_ensemble_rank_softmax_weighted_AUC0.725399',
        'oof_path': "/kaggle/input/oof-diabetic/20251205_ensemble_rank_softmax_weighted_AUC0.725399_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251205_ensemble_rank_softmax_weighted_AUC0.725399_test.npy",
    },

    {
        'name': '20251207_053254_XGB_LGB_CB_RidgeStack_alpha0.001_AUC0p730518',
        'oof_path': "/kaggle/input/oof-diabetic/20251207_053254_XGB_LGB_CB_RidgeStack_alpha0.001_AUC0p730518_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251207_053254_XGB_LGB_CB_RidgeStack_alpha0.001_AUC0p730518_test.npy",
    },
    {
        'name': '20251208_064248_XGB_LGB_CB_RidgeStack_AUC0p730575',
        'oof_path': "/kaggle/input/oof-diabetic/20251208_064248_XGB_LGB_CB_RidgeStack_AUC0p730575_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251208_064248_XGB_LGB_CB_RidgeStack_AUC0p730575_test.npy",
    },

    {
        'name': '20251208_163617_XGB_LGB_CB_RidgeStack_AUC0p730390',
        'oof_path': "/kaggle/input/oof-diabetic/20251208_163617_XGB_LGB_CB_RidgeStack_AUC0p730390_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251208_163617_XGB_LGB_CB_RidgeStack_AUC0p730390_test.npy",
    },
    {
        'name': 'preds_xgb_20251203_230310_auc0p731335',
        'oof_path': "/kaggle/input/oof-diabetic/oof_preds_xgb_20251203_230310_auc0p731335.npy",
        'test_path': "/kaggle/input/oof-diabetic/test_preds_xgb_20251203_230310_auc0p731335.npy",
    },
    {
        'name': '15122025_Ridge_3_models',
        'oof_path': "/kaggle/input/oof-diabetic/20251215_055051_Ridge_3Models_WeightedCV_fullCV0p72524_postCV0p70465_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251215_055051_Ridge_3Models_WeightedCV_fullCV0p72524_postCV0p70465_test.npy",
    },
    {
        'name': '15122025_Ensemble_3_models',
        'oof_path': "/kaggle/input/oof-diabetic/20251215_064046_3Models_Ensemble_fullCV0p72924_postCV0p69849_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251215_064046_3Models_Ensemble_fullCV0p72924_postCV0p69849_test.npy",
    },
    {
        'name': '15122025_CatBoost_3seeds',
        'oof_path': "/kaggle/input/oof-diabetic/20251215_092009_CatBoost_3Seed_WeightedCV_fullCV0p71723_postCV0p70467_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251215_092009_CatBoost_3Seed_WeightedCV_fullCV0p71723_postCV0p70467_test.npy",
    },
    {
        'name': '15122025_LGBM_Optuna_proxy_changed_CV',
        'oof_path': "/kaggle/input/oof-diabetic/20251215_072246_LightGBM_Optuna_WeightedCV_fullCV0p71983_postCV0p69887_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251215_072246_LightGBM_Optuna_WeightedCV_fullCV0p71983_postCV0p69887_test.npy",
    },
    {
        'name': '15122025_3models_Ridge_multiseed',
        'oof_path': "/kaggle/input/oof-diabetic/20251215_161744_Ridge_Ensemble_3Seed_fullCV0p71817_postCV0p70412_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251215_161744_Ridge_Ensemble_3Seed_fullCV0p71817_postCV0p70412_test.npy",
    },
    {
        'name': '15122025_XGB_3seeds',
        'oof_path': "/kaggle/input/oof-diabetic/20251215_161744_XGBoost_3Seed_fullCV0p71844_postCV0p70379_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251215_161744_XGBoost_3Seed_fullCV0p71844_postCV0p70379_test.npy",
    },
    {
        'name': '15122025_LGBM_3seeds',
        'oof_path': "/kaggle/input/oof-diabetic/20251215_161744_LightGBM_3Seed_fullCV0p71908_postCV0p70280_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251215_161744_LightGBM_3Seed_fullCV0p71908_postCV0p70280_test.npy",
    },
    {
        'name': '15122025_YDF',
        'oof_path': "/kaggle/input/oof-diabetic/20251215_171053_YDF_GBT_WeightedCV_fullCV0p71606_postCV0p69257_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251215_171053_YDF_GBT_WeightedCV_fullCV0p71606_postCV0p69257_test.npy",
    },
    {
        'name': '15122025_LGBM_Optuna_optimized',
        'oof_path': "/kaggle/input/oof-diabetic/20251215_181721_LightGBM_Optuna_WeightedCV_fullCV0p71838_postCV0p69683_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251215_181721_LightGBM_Optuna_WeightedCV_fullCV0p71838_postCV0p69683_test.npy",
    },
    {
        'name': '16122025_HistGradient',
        'oof_path': "/kaggle/input/oof-diabetic/20251215_194323_HistGB_WeightedCV_fullCV0p71390_postCV0p69018_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251215_194323_HistGB_WeightedCV_fullCV0p71390_postCV0p69018_test.npy",
    },
    {
        'name': '16122025_Gradient_Boosting',
        'oof_path': "/kaggle/input/oof-diabetic/20251215_195051_GradientBoosting_WeightedCV_fullCV0p70607_postCV0p68751_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251215_195051_GradientBoosting_WeightedCV_fullCV0p70607_postCV0p68751_test.npy",
    },
    {
        'name': '16122025_Extra_Trees',
        'oof_path': "/kaggle/input/oof-diabetic/20251215_200005_ExtraTrees_WeightedCV_fullCV0p69842_postCV0p68236_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251215_200005_ExtraTrees_WeightedCV_fullCV0p69842_postCV0p68236_test.npy",
    },
    {
        'name': '16122025_Random_Forest',
        'oof_path': "/kaggle/input/oof-diabetic/20251215_200914_RandomForest_WeightedCV_fullCV0p70122_postCV0p68173_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251215_200914_RandomForest_WeightedCV_fullCV0p70122_postCV0p68173_test.npy",
    },
    {
        'name': '16122025_NGBoost',
        'oof_path': "/kaggle/input/oof-diabetic/20251215_204426_NGBoost_WeightedCV_fullCV0p70701_postCV0p68940_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251215_204426_NGBoost_WeightedCV_fullCV0p70701_postCV0p68940_test.npy",
    },
    {
        'name': '16122025_Ridge_2Models_3seeds',
        'oof_path': "/kaggle/input/oof-diabetic/20251216_123402_Ridge_CatXGB_alpha0p0001_fullCV0p71691_postCV0p70433_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251216_123402_Ridge_CatXGB_alpha0p0001_fullCV0p71691_postCV0p70433_test.npy",
    },
    {
        'name': '16122025_Catboost_3seeds',
        'oof_path': "/kaggle/input/oof-diabetic/20251216_132847_CatBoost_3Seed_fullCV0p71722_postCV0p70479_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251216_132847_CatBoost_3Seed_fullCV0p71722_postCV0p70479_test.npy",
    },
    {
        'name': '16122025_HC_best_till_now',
        'oof_path': "/kaggle/input/oof-diabetic/ensemble_oof_greedy_rank_38of42_neg_auc0.73202_20251216_204341.npy",
        'test_path': "/kaggle/input/oof-diabetic/ensemble_test_greedy_rank_38of42_neg_auc0.73202_20251216_204341.npy",
    },
    {
        'name': '17122025_Ridge_3models_3ways_combos',
        'oof_path': "/kaggle/input/oof-diabetic/20251217_165937_Ridge_3Models_WeightedCV_2Way_fullCV0p72453_postCV0p70503_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251217_165937_Ridge_3Models_WeightedCV_2Way_fullCV0p72453_postCV0p70503_test.npy",
    },
    {
        'name': '19122025_XGB_leakage_solved',
        'oof_path': "/kaggle/input/oof-diabetic/20251219_031943_XGB_WeightedCV_fullCV0p72524_postCV0p70461_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251219_031943_XGB_WeightedCV_fullCV0p72524_postCV0p70461_test.npy",
    },
    {
        'name': '19122025_XGB_leakage_solved_v1',
        'oof_path': "/kaggle/input/oof-diabetic/20251219_033547_XGB_WeightedCV_fullCV0p72226_postCV0p70178_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251219_033547_XGB_WeightedCV_fullCV0p72226_postCV0p70178_test.npy",
    },
    {
        'name': '19122025_XGB_leakage_solved_5folds_3seeds',
        'oof_path': "/kaggle/input/oof-diabetic/20251219_035844_XGB_WeightedCV_3Seeds_fullCV0p72526_postCV0p70482_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251219_035844_XGB_WeightedCV_3Seeds_fullCV0p72526_postCV0p70482_test.npy",
    },
    {
        'name': '19122025_YDF_model',
        'oof_path': "/kaggle/input/oof-diabetic/20251219_113209_YDF_GradientBoosted_fullCV0p72266_postCV0p70104_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251219_113209_YDF_GradientBoosted_fullCV0p72266_postCV0p70104_test.npy",
    },
    # {
    #     'name': '19122025_ExtraTrees_model',
    #     'oof_path': "/kaggle/input/oof-diabetic/20251219_153604_ExtraTrees_fullCV0p70380_postCV0p68936_oof.npy",
    #     'test_path': "/kaggle/input/oof-diabetic/20251219_153604_ExtraTrees_fullCV0p70380_postCV0p68936_test.npy",
    # },
    # {
    #     'name': '19122025_MLP_Keras',
    #     'oof_path': "/kaggle/input/oof-diabetic/20251219_171401_MLP_Keras_fullCV0p71838_postCV0p70370_oof.npy",
    #     'test_path': "/kaggle/input/oof-diabetic/20251219_171401_MLP_Keras_fullCV0p71838_postCV0p70370_test.npy",
    # },
    # {
    #     'name': '19122025_MLP_Pytorch',
    #     'oof_path': "/kaggle/input/oof-diabetic/20251219_175617_MLP_PyTorch_fullCV0p71832_postCV0p70395_oof.npy",
    #     'test_path': "/kaggle/input/oof-diabetic/20251219_175617_MLP_PyTorch_fullCV0p71832_postCV0p70395_test.npy",
    # },
    # {
    #     'name': '19122025_XGB_2ways_combo',
    #     'oof_path': "/kaggle/input/oof-diabetic/20251219_181333_XGBoost_2WayCombos_fullCV0p72417_postCV0p70358_oof.npy",
    #     'test_path': "/kaggle/input/oof-diabetic/20251219_181333_XGBoost_2WayCombos_fullCV0p72417_postCV0p70358_test.npy",
    # },
    # {
    #     'name': '20122025_Deeper_MLP_Keras',
    #     'oof_path': "/kaggle/input/oof-diabetic/20251220_075534_MLP_Keras_fullCV0p71780_postCV0p70394_oof.npy",
    #     'test_path': "/kaggle/input/oof-diabetic/20251220_075534_MLP_Keras_fullCV0p71780_postCV0p70394_test.npy",
    # },
    # {
    #     'name': '20122025_MLP_Keras_multiseed',
    #     'oof_path': "/kaggle/input/oof-diabetic/20251220_084616_MLP_Keras_MultiSeed_5seeds_fullCV0p71922_postCV0p70381_oof.npy",
    #     'test_path': "/kaggle/input/oof-diabetic/20251220_084616_MLP_Keras_MultiSeed_5seeds_fullCV0p71922_postCV0p70381_test.npy",
    # },
    # {
    #     'name': '20122025_MLP_Pytorch_multiseed',
    #     'oof_path': "/kaggle/input/oof-diabetic/20251220_105039_MLP_PyTorch_5Seed_fullCV0p71964_postCV0p70428_oof.npy",
    #     'test_path': "/kaggle/input/oof-diabetic/20251220_105039_MLP_PyTorch_5Seed_fullCV0p71964_postCV0p70428_test.npy",
    # },
    # {
    #     'name': '20122025_MLP_Pytorch_421_seed',
    #     'oof_path': "/kaggle/input/oof-diabetic/20251220_105039_MLP_PyTorch_5Seed_seed421_oof.npy",
    #     'test_path': "/kaggle/input/oof-diabetic/20251220_105039_MLP_PyTorch_5Seed_seed421_test.npy",
    # },
    # {
    #     'name': '20122025_FastAI',
    #     'oof_path': "/kaggle/input/oof-diabetic/20251220_132541_FastAI_Tabular_fullCV0p72097_postCV0p70269_oof.npy",
    #     'test_path': "/kaggle/input/oof-diabetic/20251220_132541_FastAI_Tabular_fullCV0p72097_postCV0p70269_test.npy",
    # },
    # {
    #     'name': '20122025_FastAI_native',
    #     'oof_path': "/kaggle/input/oof-diabetic/20251220_134031_FastAI_Tabular_fullCV0p72106_postCV0p70079_oof.npy",
    #     'test_path': "/kaggle/input/oof-diabetic/20251220_134031_FastAI_Tabular_fullCV0p72106_postCV0p70079_test.npy",
    # },
    {
        'name': '21122025_HistGradient',
        'oof_path': "/kaggle/input/oof-diabetic/20251221_101112_HistGradientBoosting_fullCV0p72376_postCV0p70318_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251221_101112_HistGradientBoosting_fullCV0p72376_postCV0p70318_test.npy",
    },
    {
        'name': '21122025_LGBM_Optuna_optimized',
        'oof_path': "/kaggle/input/oof-diabetic/20251221_122725_LightGBM_Optuna_Poly_fullCV0p71647_postCV0p69684_oof.npy",
        'test_path': "/kaggle/input/oof-diabetic/20251221_122725_LightGBM_Optuna_Poly_fullCV0p71647_postCV0p69684_test.npy",
    },
]

# =============================================================================
# UTILITY FUNCTIONS
# =============================================================================

def torch_roc_auc(y_true, y_score):
    """GPU-accelerated AUC calculation"""
    y_true = y_true.float()
    y_score = y_score.float()
    order = torch.argsort(y_score, descending=True)
    y_sorted = y_true[order]
    pos, neg = y_sorted, 1.0 - y_sorted
    cum_pos, cum_neg = torch.cumsum(pos, dim=0), torch.cumsum(neg, dim=0)
    num_pos, num_neg = pos.sum(), neg.sum()
    if num_pos == 0 or num_neg == 0:
        return 0.5
    zero = torch.tensor([0.0], device=y_true.device)
    tpr = torch.cat([zero, cum_pos / num_pos])
    fpr = torch.cat([zero, cum_neg / num_neg])
    return float(torch.sum((fpr[1:] - fpr[:-1]) * (tpr[1:] + tpr[:-1]) * 0.5))

def convert_to_ranks(predictions):
    """Convert predictions to ranks [0, 1]"""
    n = predictions.size(0)
    ranks = torch.argsort(torch.argsort(predictions)).float()
    return ranks / (n - 1) if n > 1 else ranks

def resolve_start_model(start_model, model_names, model_aucs):
    """
    Resolve the starting model index from user input.
    
    Args:
        start_model: None (auto), str (model name), or int (model index)
        model_names: List of model names
        model_aucs: List of model AUC scores (post-cutoff)
        
    Returns:
        int: Index of the starting model
    """
    if start_model is None:
        idx = int(np.argmax(model_aucs))
        print(f"Auto-selected: {model_names[idx]} (AUC: {model_aucs[idx]:.6f})")
        return idx
    
    elif isinstance(start_model, int):
        if start_model < 0 or start_model >= len(model_names):
            raise ValueError(f"Index {start_model} out of range [0, {len(model_names)-1}]")
        print(f"User-selected (index {start_model}): {model_names[start_model]} "
              f"(AUC: {model_aucs[start_model]:.6f})")
        return start_model
    
    elif isinstance(start_model, str):
        if start_model not in model_names:
            matches = [i for i, name in enumerate(model_names) if start_model.lower() in name.lower()]
            if len(matches) == 1:
                idx = matches[0]
                print(f"User-selected (partial match): {model_names[idx]} (AUC: {model_aucs[idx]:.6f})")
                return idx
            elif len(matches) > 1:
                raise ValueError(f"'{start_model}' matches multiple models: "
                               f"{[model_names[i] for i in matches]}")
            else:
                raise ValueError(f"'{start_model}' not found in model list")
        idx = model_names.index(start_model)
        print(f"User-selected: {model_names[idx]} (AUC: {model_aucs[idx]:.6f})")
        return idx
    
    else:
        raise ValueError(f"start_model must be None, int, or str. Got: {type(start_model)}")

def greedy_hill_climbing(oof_preds, y_true_post, post_mask, names, 
                        weight_min, weight_max, weight_step, tolerance,
                        start_idx, y_true_full):
    """Greedy forward selection optimizing post-cutoff AUC"""
    n = len(oof_preds)
    aucs = [torch_roc_auc(y_true_post, p[post_mask]) for p in oof_preds]
    
    ensemble_full = oof_preds[start_idx].clone()
    ensemble_post = ensemble_full[post_mask]
    ensemble_auc = aucs[start_idx]
    
    used = [start_idx]
    remaining = [i for i in range(n) if i != start_idx]
    weights_map = {start_idx: 1.0}
    
    history = [{
        "iteration": 0, "model": names[start_idx],
        "weight": 1.0, "auc": ensemble_auc
    }]
    
    print(f"Start: {names[start_idx]} | Post-Cutoff AUC: {ensemble_auc:.6f}")
    print("-"*80)
    
    weights = torch.arange(weight_min, weight_max + 1e-12, weight_step, 
                          device=post_mask.device).float()
    
    it = 0
    while remaining:
        it += 1
        best_imp, best_info = 0.0, None
        best_combined_full, best_auc_val, best_new_weights = None, None, None
        
        for idx in remaining:
            cand_full = oof_preds[idx]
            cand_post = cand_full[post_mask]
            
            # Test all weights
            wcol = weights.view(-1, 1)
            combined_post = ensemble_post.unsqueeze(0) * (1.0 - wcol) + \
                          cand_post.unsqueeze(0) * wcol
            combined_post = combined_post.clamp_(0.0, 1.0)
            
            # Find best weight for this candidate
            best_local_imp, best_local_w, best_local_auc = None, None, None
            for wi in range(combined_post.size(0)):
                auc_val = torch_roc_auc(y_true_post, combined_post[wi])
                imp = auc_val - ensemble_auc
                if best_local_imp is None or imp > best_local_imp:
                    best_local_imp, best_local_w, best_local_auc = imp, float(weights[wi]), auc_val
            
            if best_local_imp and best_local_imp > best_imp:
                best_imp = best_local_imp
                best_info = (idx, best_local_w, names[idx])
                best_auc_val = best_local_auc
                best_combined_full = ensemble_full * (1.0 - best_local_w) + cand_full * best_local_w
                best_combined_full = best_combined_full.clamp_(0.0, 1.0)
                new_w = {k: v*(1.0 - best_local_w) for k, v in weights_map.items()}
                new_w[idx] = best_local_w
                best_new_weights = new_w
        
        if best_imp < tolerance or best_info is None:
            break
        
        # Accept best addition
        add_idx, w, add_name = best_info
        ensemble_full = best_combined_full
        ensemble_post = ensemble_full[post_mask]
        ensemble_auc = best_auc_val
        weights_map = best_new_weights
        used.append(add_idx)
        remaining.remove(add_idx)
        
        history.append({
            "iteration": it, "model": add_name,
            "weight": w, "auc": float(ensemble_auc)
        })
        
        print(f"[{it:2d}] {add_name:<50s} w={w:+.4f} | AUC: {ensemble_auc:.6f} (+{best_imp:.6f})")
    
    return ensemble_full, weights_map, history, used

# =============================================================================
# STEP 1: LOAD DATA
# =============================================================================

print("STEP 1: LOADING DATA")
print("-"*80)

target_df = pd.read_csv(TARGET_PATH)
test_df = pd.read_csv(TEST_PATH)
y_true_np = target_df[TARGET_COLUMN].values.astype(np.float32)
post_mask_np = (target_df['id'] >= CUTOFF_ID).values if 'id' in target_df.columns \
               else (target_df.index >= CUTOFF_ID).values

n_total, n_post = len(y_true_np), post_mask_np.sum()
print(f"Total samples: {n_total:,} | Post-cutoff: {n_post:,} ({n_post/n_total*100:.1f}%)")
print(f"Test samples: {len(test_df):,}")

# Load models
oof_list_np, test_list_np, model_names = [], [], []
for i, m in enumerate(MODELS, 1):
    try:
        oof_list_np.append(np.load(m['oof_path']).astype(np.float32))
        test_list_np.append(np.load(m['test_path']).astype(np.float32))
        model_names.append(m['name'])
    except Exception as e:
        print(f"  Warning: Failed to load {m['name']}: {e}")

print(f"Loaded {len(model_names)} models\n")

# =============================================================================
# STEP 2: HILL CLIMBING SELECTION
# =============================================================================

print("="*80)
print("STEP 2: HILL CLIMBING MODEL SELECTION")
print("="*80)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Optimization target: Post-cutoff AUC (ID >= {CUTOFF_ID})")
print(f"Blending: {HC_BLENDING_METHOD.upper()} | Weights: [{HC_WEIGHT_MIN}, {HC_WEIGHT_MAX}]\n")

# Setup tensors
y_true_full = torch.from_numpy(y_true_np).to(device)
post_mask = torch.from_numpy(post_mask_np).to(device)
y_true_post = y_true_full[post_mask]

oof_list_raw = [torch.from_numpy(x).to(device) for x in oof_list_np]
test_list_raw = [torch.from_numpy(x).to(device) for x in test_list_np]

# Apply rank transformation
if HC_BLENDING_METHOD == "rank":
    oof_list = [convert_to_ranks(oof) for oof in oof_list_raw]
    test_list = [convert_to_ranks(test) for test in test_list_raw]
else:
    oof_list = oof_list_raw
    test_list = test_list_raw

# Calculate individual AUCs
model_auc_post = [torch_roc_auc(y_true_post, oof[post_mask]) for oof in oof_list]

# Resolve starting model
start_idx = resolve_start_model(HC_START_MODEL, model_names, model_auc_post)
print()

# Run hill climbing
start_time = time.time()
ensemble_oof, final_weights, history, selected_indices = greedy_hill_climbing(
    oof_list, y_true_post, post_mask, model_names,
    HC_WEIGHT_MIN, HC_WEIGHT_MAX, HC_WEIGHT_STEP, HC_TOLERANCE,
    start_idx, y_true_full
)
hc_time = time.time() - start_time

hc_auc = torch_roc_auc(y_true_post, ensemble_oof[post_mask])
print("\n" + "-"*80)
print(f"Hill Climbing Complete: {len(selected_indices)} models selected in {hc_time:.1f}s")
print(f"HC Post-Cutoff AUC: {hc_auc:.6f}\n")

# Extract selected models
hc_selected_names = [model_names[i] for i in selected_indices]

# =============================================================================
# STEP 3: RIDGE ENSEMBLE (FROM HC-SELECTED MODELS ONLY)
# =============================================================================

print("="*80)
print("STEP 3: RIDGE ENSEMBLE")
print("="*80)

# Rank HC-selected models by post-cutoff AUC
hc_aucs = [model_auc_post[i] for i in selected_indices]
sorted_hc_indices = [selected_indices[i] for i in np.argsort(hc_aucs)[::-1]]

print(f"Configuration: TOP_N={RIDGE_TOP_N}, ALPHA={RIDGE_ALPHA}")
print(f"Source: {len(selected_indices)} models from Hill Climbing\n")
print("HC-selected models ranked by post-cutoff AUC:")
for i, idx in enumerate(sorted_hc_indices[:min(20, len(sorted_hc_indices))]):
    marker = "✓" if i < RIDGE_TOP_N else " "
    print(f"{marker} {i+1:2d}. {model_names[idx]:<50s} AUC: {model_auc_post[idx]:.6f}")

if len(sorted_hc_indices) > 20:
    print(f"... and {len(sorted_hc_indices) - 20} more")

# Select top-N from HC models
ridge_top_n = min(RIDGE_TOP_N, len(sorted_hc_indices))
ridge_indices = sorted_hc_indices[:ridge_top_n]
ridge_names = [model_names[i] for i in ridge_indices]

print(f"\nUsing top-{ridge_top_n} models from HC selection for Ridge ensemble")

# Convert to ranks for Ridge
print(f"Converting to ranks...")
oof_ranks = [convert_to_ranks(oof_list_raw[i]).cpu().numpy() for i in ridge_indices]
test_ranks = [convert_to_ranks(test_list_raw[i]).cpu().numpy() for i in ridge_indices]

X_oof_full = np.column_stack(oof_ranks)
X_test = np.column_stack(test_ranks)
X_oof_post = X_oof_full[post_mask_np]
y_oof_post = y_true_np[post_mask_np]

print(f"Feature matrix: {X_oof_post.shape}\n")

# Fit Ridge
print(f"Fitting Ridge (alpha={RIDGE_ALPHA})...")
ridge = Ridge(alpha=RIDGE_ALPHA, fit_intercept=True, random_state=42)
ridge.fit(X_oof_post, y_oof_post)

ridge_oof = np.clip(ridge.predict(X_oof_full), 0, 1)
ridge_test = np.clip(ridge.predict(X_test), 0, 1)

# Evaluate
ridge_oof_t = torch.from_numpy(ridge_oof.astype(np.float32)).to(device)
ridge_auc_post = torch_roc_auc(y_true_post, ridge_oof_t[post_mask])
ridge_auc_full = torch_roc_auc(y_true_full, ridge_oof_t)

print(f"\nRidge Results:")
print(f"  Post-Cutoff AUC: {ridge_auc_post:.6f}")
print(f"  Full CV AUC:     {ridge_auc_full:.6f}\n")

# =============================================================================
# STEP 4: DIVERSITY ANALYSIS
# =============================================================================

print("="*80)
print("STEP 4: DIVERSITY ANALYSIS")
print("="*80)

# Calculate correlation matrix
corr_matrix = np.corrcoef(X_oof_post.T)

print("\nRidge Model Weights:")
sorted_coefs = sorted(zip(ridge_names, ridge.coef_), key=lambda x: abs(x[1]), reverse=True)
for name, coef in sorted_coefs[:15]:
    print(f"  {name:<50s} {coef:>8.4f}")
print(f"  {'...':<50s}")
print(f"  Intercept: {ridge.intercept_:>8.4f}\n")

# Performance stats
ridge_aucs = [model_auc_post[i] for i in ridge_indices]
print("Performance Statistics:")
print(f"  Mean AUC:   {np.mean(ridge_aucs):.6f}")
print(f"  Std AUC:    {np.std(ridge_aucs):.6f}")
print(f"  Min AUC:    {np.min(ridge_aucs):.6f}")
print(f"  Max AUC:    {np.max(ridge_aucs):.6f}")
print(f"  Range:      {np.max(ridge_aucs) - np.min(ridge_aucs):.6f}\n")

# Correlation stats
upper_tri = corr_matrix[np.triu_indices_from(corr_matrix, k=1)]
print("Correlation Statistics:")
print(f"  Mean correlation:   {np.mean(upper_tri):.4f}")
print(f"  Median correlation: {np.median(upper_tri):.4f}")
print(f"  Min correlation:    {np.min(upper_tri):.4f}")
print(f"  Max correlation:    {np.max(upper_tri):.4f}")
print(f"  Std correlation:    {np.std(upper_tri):.4f}\n")

# Most diverse pairs
flat_indices = np.argsort(upper_tri)
print("Most Diverse Model Pairs (lowest correlation):")
for idx in flat_indices[:5]:
    i, j = np.triu_indices_from(corr_matrix, k=1)
    print(f"  {ridge_names[i[idx]][:40]:<40s} <-> {ridge_names[j[idx]][:40]:<40s} r={upper_tri[idx]:.4f}")

print("\nMost Similar Model Pairs (highest correlation):")
for idx in flat_indices[-5:][::-1]:
    i, j = np.triu_indices_from(corr_matrix, k=1)
    print(f"  {ridge_names[i[idx]][:40]:<40s} <-> {ridge_names[j[idx]][:40]:<40s} r={upper_tri[idx]:.4f}")

# HC vs Ridge overlap
print(f"\nModel Selection:")
print(f"  HC selected:     {len(selected_indices)} models")
print(f"  Ridge used:      {ridge_top_n} models (top-{RIDGE_TOP_N} from HC)")
print(f"  Source:          100% from HC selection\n")

# =============================================================================
# STEP 5: SAVE RESULTS
# =============================================================================

print("="*80)
print("STEP 5: SAVING RESULTS")
print("="*80)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")

# Save submission
pd.DataFrame({
    "id": test_df['id'],
    TARGET_COLUMN: ridge_test
}).to_csv(os.path.join(OUT_DIR, "submission.csv"), index=False)
print("✓ Saved: submission.csv")

# Save OOF predictions
np.save(os.path.join(OUT_DIR, f"ridge_oof_{ts}.npy"), ridge_oof)
pd.DataFrame({
    "id": target_df['id'] if 'id' in target_df.columns else target_df.index,
    TARGET_COLUMN: ridge_oof
}).to_csv(os.path.join(OUT_DIR, f"ridge_oof_{ts}.csv"), index=False)
print(f"✓ Saved: ridge_oof_{ts}.csv")

# Save HC results
np.save(os.path.join(OUT_DIR, f"hc_oof_{ts}.npy"), ensemble_oof.cpu().numpy())
print(f"✓ Saved: hc_oof_{ts}.npy")

# Save summary
with open(os.path.join(OUT_DIR, f"solution_summary_{ts}.txt"), "w") as f:
    f.write("1ST PLACE SOLUTION SUMMARY\n")
    f.write("="*80 + "\n\n")
    
    f.write("CONFIGURATION\n")
    f.write("-"*80 + "\n")
    f.write(f"Cutoff ID: {CUTOFF_ID}\n")
    f.write(f"Hill Climbing: {HC_BLENDING_METHOD} blending, {len(selected_indices)} models selected\n")
    f.write(f"Ridge Ensemble: Top-{ridge_top_n} from HC models, alpha={RIDGE_ALPHA}\n\n")
    
    f.write("RESULTS\n")
    f.write("-"*80 + "\n")
    f.write(f"Hill Climbing AUC:      {hc_auc:.6f}\n")
    f.write(f"Ridge Post-Cutoff AUC:  {ridge_auc_post:.6f}\n")
    f.write(f"Ridge Full CV AUC:      {ridge_auc_full:.6f}\n\n")
    
    f.write("RIDGE MODEL WEIGHTS\n")
    f.write("-"*80 + "\n")
    for name, coef in sorted_coefs:
        f.write(f"{name:<50s} {coef:>8.4f}\n")
    f.write(f"Intercept: {ridge.intercept_:>8.4f}\n\n")
    
    f.write("DIVERSITY METRICS\n")
    f.write("-"*80 + "\n")
    f.write(f"Mean AUC:           {np.mean(ridge_aucs):.6f}\n")
    f.write(f"AUC Range:          {np.max(ridge_aucs) - np.min(ridge_aucs):.6f}\n")
    f.write(f"Mean Correlation:   {np.mean(upper_tri):.4f}\n")
    f.write(f"HC Models Used:     {ridge_top_n}/{len(selected_indices)} (top-{RIDGE_TOP_N})\n")

print(f"✓ Saved: solution_summary_{ts}.txt")

print("\n" + "="*80)
print("PIPELINE COMPLETE!")
print("="*80)
print(f"\n📊 Final Model Performance:")
print(f"   Post-Cutoff CV AUC: {ridge_auc_post:.6f}")
print(f"\n📁 Submission file: submission.csv")
print(f"   Ready to submit!\n")